# Amharic Whisper Large — Ray Train + LoRA (Multi-GPU)

Fine-tunes **`openai/whisper-large-v3`** on all 4 Leyu Amharic dialect datasets  
plus ALFFA using **Ray Train** for data-parallel multi-GPU training and **PEFT LoRA**  
for parameter-efficient fine-tuning.

| Dataset | Dialect | Rows | Speakers |
|---|---|---|---|
| `leyu-amharic/leyu-amharic-gojjam-dialect` | Gojjam | ~10,575 | 82 |
| `leyu-amharic/leyu-amharic-gonder-dialect` | Gonder | ~8,994 | 43 |
| `leyu-amharic/leyu-amharic-wello-dialect`  | Wello  | 4,862   | 51 |
| `leyu-amharic/leyu-amharic-shewa-dialect`  | Shewa  | 2,594   | 52 |

**Hardware targets**: 1× Nvidia H200 (single-node, large batch) **or** 2–4× Nvidia L40S (multi-GPU DDP via Ray)  
**Strategy**:
- `whisper-large-v3` loaded in **bf16**; LoRA injected on encoder *and* decoder attention projections
- Ray `TorchTrainer` handles DDP worker launch, gradient sync, and checkpoint management
- LoRA rank **r=32**, alpha=64 — keeps trainable params to ~40 M out of 1.5 B


## 0. Environment Setup

In [1]:
# Fix audio backend — use soundfile, avoid torchcodec conflicts
import os, subprocess, sys

os.environ["FORCE_SOUNDFILE"]    = "1"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchcodec", "-y"],
               capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "soundfile", "librosa", "-q"],
               capture_output=True)

print("✅ Audio backend ready")


✅ Audio backend ready


In [2]:
# Install all dependencies — pinned to latest stable versions as of June 2026
import subprocess, sys

pkgs = [
    # Core ML
    "torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124",
    # HuggingFace stack
    "transformers==5.6.2",
    "datasets",
    "accelerate",
    "peft==0.19.1",
    "evaluate",
    "jiwer",
    # Ray Train (includes ray[train] extras)
    "ray[train]==2.55.1",
    # Audio
    "soundfile",
    "librosa",
]

for pkg in pkgs:
    ret = subprocess.run(
        f"{sys.executable} -m pip install -q {pkg}",
        shell=True, capture_output=True, text=True
    )
    if ret.returncode != 0:
        print(f"  ⚠️  Error installing {pkg.split()[0]}:\n{ret.stderr[:300]}")
    else:
        print(f"  ✅ {pkg.split()[0]}")

print("\n✅ All packages installed")


  ✅ torch
  ✅ transformers==5.6.2
  ✅ datasets
  ✅ accelerate
  ✅ peft==0.19.1
  ✅ evaluate
  ✅ jiwer
  ✅ ray[train]==2.55.1
  ✅ soundfile
  ✅ librosa

✅ All packages installed


## 1. Configuration

In [3]:
import os
from pathlib import Path

# ── Datasets ───────────────────────────────────────────────────────────────
LEYU_DATASETS = [
    "leyu-amharic/leyu-amharic-gojjam-dialect",
    "leyu-amharic/leyu-amharic-gonder-dialect",
    "leyu-amharic/leyu-amharic-wello-dialect",
    "leyu-amharic/leyu-amharic-shewa-dialect",
]
ALFFA_DATASET = "hadamard-2/alffa-amharic"

# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME = "openai/whisper-large-v3"   

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_R           = 32       
LORA_ALPHA       = 64       
LORA_DROPOUT     = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj"]

# ── Ray Training ───────────────────────────────────────────────────────────
NUM_WORKERS       = 2        
USE_GPU           = True
CPUS_PER_WORKER   = 4

# ── Optimized Training Hyperparameters for H200 NVL ────────────────────────
EPOCHS            = 10
PER_GPU_BATCH     = 64       # OPTIMIZED: Cranked up from 8 to feed the 150GB VRAM
GRAD_ACCUM        = 1        # OPTIMIZED: Set to 1 because your raw batch size is now huge
LEARNING_RATE     = 2e-4     # OPTIMIZED: Scaled slightly higher to match larger batch size
WARMUP_STEPS      = 200
EVAL_STEPS        = 500      # OPTIMIZED: Evaluates roughly once per epoch instead of 5 times
SAVE_STEPS        = 500      # OPTIMIZED: Aligned with evaluation steps
MAX_TEXT_CHARS    = 350

# ── Data splits ────────────────────────────────────────────────────────────
VAL_RATIO  = 0.10
TEST_RATIO = 0.10
SEED       = 42

# ── Hub ────────────────────────────────────────────────────────────────────
PUSH_TO_HUB  = False
HUB_MODEL_ID = "your-username/whisper-large-amharic-lora"

# ── SAFE STORAGE REDIRECTION (Bypassing Full /home & /storage) ─────────────
MY_USERNAME = os.environ.get("USER", "my_workspace")
SAFE_NVME_ROOT = f"/tmp/{MY_USERNAME}_cache"

# Force Hugging Face and Jupyter to use the NVMe drive in the background
os.environ["IPYTHONDIR"] = os.path.join(SAFE_NVME_ROOT, "ipython_history")
os.environ["HF_HOME"] = os.path.join(SAFE_NVME_ROOT, "huggingface")
os.environ["HF_DATASETS_CACHE"] = os.path.join(SAFE_NVME_ROOT, "huggingface", "datasets")
os.environ["RAY_AIR_LOCAL_DIR"] = os.path.join(SAFE_NVME_ROOT, "ray_results")
os.environ["RAY_RESULTS_DIR"] = os.path.join(SAFE_NVME_ROOT, "ray_results")

# Update your active output paths to the safe root
OUTPUT_DIR  = os.path.join(SAFE_NVME_ROOT, "whisper-large-amharic-lora")
LOG_DIR     = os.path.join(SAFE_NVME_ROOT, "whisper-large-amharic-lora-logs")
DATASET_DIR = os.path.join(SAFE_NVME_ROOT, "whisper-amharic-dataset")   

# Build directories safely on the NVMe partition
for d in [OUTPUT_DIR, LOG_DIR, DATASET_DIR]:
    os.makedirs(d, exist_ok=True)

eff_batch = PER_GPU_BATCH * NUM_WORKERS * GRAD_ACCUM

print(f"🚀 Config optimized for dual H200 execution!")
print(f"   Model          : {MODEL_NAME}")
print(f"   Ray workers    : {NUM_WORKERS}  (GPUs)")
print(f"   Effective batch: {eff_batch}")
print(f"   Storage Path   : {SAFE_NVME_ROOT}")

🚀 Config optimized for dual H200 execution!
   Model          : openai/whisper-large-v3
   Ray workers    : 2  (GPUs)
   Effective batch: 128
   Storage Path   : /tmp/s8345520_cache


## 2. Check GPUs

In [4]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU detected. Check your runtime settings.")

n_gpus = torch.cuda.device_count()
print(f"  GPUs available : {n_gpus}")
for i in range(n_gpus):
    name  = torch.cuda.get_device_name(i)
    vram  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"    GPU {i}: {name}  |  {vram:.1f} GB VRAM")

if n_gpus < NUM_WORKERS:
    print(f"⚠️  NUM_WORKERS={NUM_WORKERS} but only {n_gpus} GPU(s) visible. "
          f"Adjusting NUM_WORKERS to {n_gpus}.")
    NUM_WORKERS = n_gpus
    
print(f"\n✅ Will use {NUM_WORKERS} GPU worker(s)")


  GPUs available : 2
    GPU 0: NVIDIA H200 NVL  |  150.1 GB VRAM
    GPU 1: NVIDIA H200 NVL  |  150.1 GB VRAM

✅ Will use 2 GPU worker(s)


## 3. Imports

In [5]:
import random, io, json, gc
from dataclasses import dataclass
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import torch
import soundfile as sf
import evaluate as hf_evaluate

from datasets import load_dataset, concatenate_datasets, DatasetDict, Audio

from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    PeftModel,
)

import ray
import ray.train
from ray.train import RunConfig, ScalingConfig, CheckpointConfig
from ray.train.huggingface.transformers import prepare_trainer, RayTrainReportCallback
from ray.train.torch import TorchTrainer as RayTorchTrainer

print(" Imports complete")
print(f"   torch        : {torch.__version__}")
import transformers, peft
print(f"   transformers : {transformers.__version__}")
print(f"   peft         : {peft.__version__}")
print(f"   ray          : {ray.__version__}")


 Imports complete
   torch        : 2.6.0+cu124
   transformers : 5.6.2
   peft         : 0.19.1
   ray          : 2.55.1


## 4. Amharic Text Normalization

Amharic has orthographic variants for phonetically identical characters  
(e.g. ሐ/ሀ/ኀ all represent the same sound in modern usage). We normalize  
before WER/CER computation to avoid penalizing correct phonetic predictions.


In [6]:
AMHARIC_CHAR_MAP = {
    # ሐ row → ሀ row
    "ሐ":"ሀ","ሑ":"ሁ","ሒ":"ሂ","ሓ":"ሃ","ሔ":"ሄ","ሕ":"ህ","ሖ":"ሆ",
    # ኀ row → ሀ row
    "ኀ":"ሀ","ኁ":"ሁ","ኂ":"ሂ","ኃ":"ሃ","ኄ":"ሄ","ኅ":"ህ","ኆ":"ሆ",
    # ዐ row → አ row (glottal stop)
    "ዐ":"አ","ዑ":"ኡ","ዒ":"ኢ","ዓ":"አ","ዔ":"ኤ","ዕ":"እ","ዖ":"ኦ",
    # ሰ row → ሠ
    "ሰ":"ሠ","ሱ":"ሡ","ሲ":"ሢ","ሳ":"ሣ","ሴ":"ሤ","ስ":"ሥ","ሶ":"ሦ",
    # ፀ row → ጸ
    "ፀ":"ጸ","ፁ":"ጹ","ፂ":"ጺ","ፃ":"ጻ","ፄ":"ጼ","ፅ":"ጽ","ፆ":"ጾ",
}
PUNCT = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~።፣፤፥፦'

def normalize_amharic(text: str) -> str:
    for src, tgt in AMHARIC_CHAR_MAP.items():
        text = text.replace(src, tgt)
    text = text.translate(str.maketrans("", "", PUNCT))
    return " ".join(text.split()).strip()

test = "ዛሬ ምግብ ቤቱ ዘግቷል። ሐሙስ ዐርብ"
print(f"Original  : {test}")
print(f"Normalized: {normalize_amharic(test)}")
print("✅ Normalization OK")


Original  : ዛሬ ምግብ ቤቱ ዘግቷል። ሐሙስ ዐርብ
Normalized: ዛሬ ምግብ ቤቱ ዘግቷል ሀሙሥ አርብ
✅ Normalization OK


## 5. Load All Dialect Datasets from HuggingFace

In [7]:
print("Loading Leyu dialect datasets...\n")

all_dialects = []
for repo_id in LEYU_DATASETS:
    print(f"  Loading {repo_id}...")
    ds = load_dataset(repo_id, split="train")
    ds = ds.cast_column("audio", Audio(sampling_rate=16_000, decode=False))
    all_dialects.append(ds)
    n_spk = len(set(ds["speaker_id"]))
    print(f"    → {len(ds):,} examples | {n_spk} speakers")

print("\n📥 Loading ALFFA Amharic...")
alffa = load_dataset(ALFFA_DATASET, split="train")
alffa = alffa.cast_column("audio", Audio(sampling_rate=16_000, decode=False))
alffa = alffa.rename_column("transcript", "text")
alffa = alffa.map(lambda x: {"dialect": "standard", "gender": "unknown"})
alffa = alffa.remove_columns(["utterance_id", "split"])
print(f"    → {len(alffa):,} examples")

combined = concatenate_datasets([*all_dialects, alffa])

print(f"\n✅ Combined:")
print(f"   Total examples : {len(combined):,}")
print(f"   Dialects       : {set(combined['dialect'])}")
print(f"   Total speakers : {len(set(combined['speaker_id']))}")


Loading Leyu dialect datasets...

  Loading leyu-amharic/leyu-amharic-gojjam-dialect...
    → 10,575 examples | 50 speakers
  Loading leyu-amharic/leyu-amharic-gonder-dialect...
    → 8,994 examples | 43 speakers
  Loading leyu-amharic/leyu-amharic-wello-dialect...
    → 4,862 examples | 26 speakers
  Loading leyu-amharic/leyu-amharic-shewa-dialect...
    → 2,594 examples | 18 speakers

📥 Loading ALFFA Amharic...
    → 10,875 examples

✅ Combined:
   Total examples : 37,900
   Dialects       : {'gonder', 'wello', 'shewa', 'gojjam', 'standard'}
   Total speakers : 11001


## 6. Speaker-Independent Train / Val / Test Split

Split by **speaker** (not by row) to ensure the model is evaluated on  
completely unseen voices — this is the correct evaluation protocol for ASR.


In [8]:
speakers = defaultdict(list)
for i, row in enumerate(combined):
    speakers[row["speaker_id"]].append(i)

spk_ids = list(speakers.keys())
random.Random(SEED).shuffle(spk_ids)
n = len(spk_ids)

n_test = max(1, int(n * TEST_RATIO))
n_val  = max(1, int(n * VAL_RATIO))

test_spk  = set(spk_ids[:n_test])
val_spk   = set(spk_ids[n_test:n_test + n_val])
train_spk = set(spk_ids[n_test + n_val:])

train_idx = [i for s, idxs in speakers.items() if s in train_spk for i in idxs]
val_idx   = [i for s, idxs in speakers.items() if s in val_spk   for i in idxs]
test_idx  = [i for s, idxs in speakers.items() if s in test_spk  for i in idxs]

dataset = DatasetDict({
    "train"     : combined.select(train_idx),
    "validation": combined.select(val_idx),
    "test"      : combined.select(test_idx),
})

print("✅ Speaker-independent split:")
for split, ds in dataset.items():
    spks = len(set(ds["speaker_id"]))
    print(f"   {split:12s}: {len(ds):5,} examples | {spks} speakers")


✅ Speaker-independent split:
   train       : 30,028 examples | 8801 speakers
   validation  : 3,981 examples | 1100 speakers
   test        : 3,891 examples | 1100 speakers


In [9]:
# Optional: filter very long transcripts
before = {k: len(v) for k, v in dataset.items()}
dataset = dataset.filter(lambda x: len(x["text"]) <= MAX_TEXT_CHARS)
after  = {k: len(v) for k, v in dataset.items()}

print("✅ Filtered long transcripts:")
for split in dataset:
    dropped = before[split] - after[split]
    pct     = dropped / before[split] * 100 if before[split] > 0 else 0
    print(f"   {split:12s}: {after[split]:,} kept  ({dropped:,} dropped, {pct:.0f}%)")


✅ Filtered long transcripts:
   train       : 24,514 kept  (5,514 dropped, 18%)
   validation  : 3,290 kept  (691 dropped, 17%)
   test        : 2,874 kept  (1,017 dropped, 26%)


## 7. Load Processor & Extract Features

We pre-compute log-Mel spectrogram features for all splits and cache them to disk.  
This happens **once** on the driver — Ray workers then load the cached dataset,  
avoiding repeated audio decoding during training.


In [10]:
print(f"📦 Loading {MODEL_NAME} processor...")
processor = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language="amharic",
    task="transcribe",
)

MAX_LABEL_LENGTH = 448

def prepare_batch(batch):
    audio_data = batch["audio"]

    if isinstance(audio_data, dict) and "bytes" in audio_data:
        audio_array, sr = sf.read(io.BytesIO(audio_data["bytes"]))
        if audio_array.ndim > 1:
            audio_array = audio_array.mean(axis=1)
        if sr != 16_000:
            import librosa
            audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16_000)
    else:
        audio_array = audio_data["array"]

    batch["input_features"] = processor.feature_extractor(
        audio_array,
        sampling_rate=16_000,
        return_tensors="pt",
    ).input_features[0]

    batch["labels"] = processor.tokenizer(
        normalize_amharic(batch["text"]),
        max_length=MAX_LABEL_LENGTH,
        truncation=True,
    ).input_ids

    return batch

print("⚙️  Extracting features (this can take 10-30 min for the full dataset)...")
dataset = dataset.map(
    prepare_batch,
    remove_columns=["text", "audio", "dialect", "speaker_id", "gender"],
    num_proc=4,
    desc="Extracting log-Mel features",
)

# Save pre-processed dataset to disk so Ray workers can load it fast
dataset.save_to_disk(DATASET_DIR)
print(f"\n✅ Features extracted and saved to: {DATASET_DIR}")
for split in dataset:
    print(f"   {split:12s}: {len(dataset[split]):,} examples")


📦 Loading openai/whisper-large-v3 processor...
⚙️  Extracting features (this can take 10-30 min for the full dataset)...


Saving the dataset (0/76 shards):   0%|          | 0/24514 [00:00<?, ? examples/s]

Saving the dataset (0/11 shards):   0%|          | 0/3290 [00:00<?, ? examples/s]

Saving the dataset (0/9 shards):   0%|          | 0/2874 [00:00<?, ? examples/s]


✅ Features extracted and saved to: /tmp/s8345520_cache/whisper-amharic-dataset
   train       : 24,514 examples
   validation  : 3,290 examples
   test        : 2,874 examples


## 8. Data Collator & Metrics

In [11]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: WhisperProcessor
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch  = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        lids = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        if (lids[:, 0] == self.decoder_start_token_id).all().cpu().item():
            lids = lids[:, 1:]

        batch["labels"] = lids
        return batch

wer_metric = hf_evaluate.load("wer")
cer_metric = hf_evaluate.load("cer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = [normalize_amharic(s) for s in
                 processor.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)]
    label_str = [normalize_amharic(s) for s in
                 processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]

    return {
        "wer": round(wer_metric.compute(predictions=pred_str, references=label_str) * 100, 2),
        "cer": round(cer_metric.compute(predictions=pred_str, references=label_str) * 100, 2),
    }

print("✅ Data collator and metrics ready")


✅ Data collator and metrics ready


## 9. LoRA Configuration

LoRA injects low-rank update matrices into the attention projections.  
With `r=32`, Whisper Large goes from **1.54 B** total params down to  
**~40 M** trainable — roughly 2.6% — while retaining competitive accuracy.

Key choices:
- **`task_type=SEQ_2_SEQ_LM`** — correct for encoder-decoder models like Whisper  
- **`modules_to_save=["embed_tokens", "lm_head"]`** — these embeddings handle the Amharic  
  script tokens added by the Whisper tokenizer; they must be fully fine-tuned, not LoRA'd  
- **`bias="none"`** — standard practice; adding bias LoRA rarely helps


In [12]:
def build_lora_model(model_name: str) -> WhisperForConditionalGeneration:
    """Load Whisper Large and inject LoRA adapters."""

    base = WhisperForConditionalGeneration.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,   # bf16 on H200 / L40S
    )

    # Required for gradient checkpointing + generation
    base.config.use_cache = False
    base.generation_config.language = "amharic"
    base.generation_config.task     = "transcribe"
    base.generation_config.forced_decoder_ids = None

    lora_cfg = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        # Fully train embedding and LM head so Amharic tokens are learned
        modules_to_save=["embed_tokens", "lm_head"],
        inference_mode=False,
    )

    model = get_peft_model(base, lora_cfg)
    model.enable_input_require_grads()   # needed for gradient checkpointing with PEFT
    model.gradient_checkpointing_enable()

    total     = sum(p.numel() for p in model.parameters()) / 1e6
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    print(f"   LoRA model ready")
    print(f"   Total params    : {total:.1f} M")
    print(f"   Trainable params: {trainable:.1f} M  ({100*trainable/total:.1f}%)")

    return model

# Quick build test on driver (no GPU needed for shape check)
print("🔍 Verifying LoRA model structure...")
_m = build_lora_model(MODEL_NAME)
_m.print_trainable_parameters()
del _m
gc.collect()
print("✅ LoRA config verified")


🔍 Verifying LoRA model structure...


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

/home/stud_homes/s8345520/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


   LoRA model ready
   Total params    : 1641.3 M
   Trainable params: 97.8 M  (6.0%)
trainable params: 97,845,760 || all params: 1,641,336,320 || trainable%: 5.9613
✅ LoRA config verified


## 10. Ray Train Worker Function

`train_loop_per_worker` runs inside each Ray worker process (one per GPU).  
Each worker:
1. Loads the pre-processed dataset shard assigned to it by Ray's `prepare_data_loader`
2. Builds the LoRA model and wraps it with DDP via `ray.train.torch.prepare_model`
3. Trains using HuggingFace `Seq2SeqTrainer` — all gradient sync is handled by Ray + PyTorch DDP
4. Reports metrics to the Ray Train controller and saves checkpoints

> **Why `Seq2SeqTrainer` inside Ray instead of raw training loop?**  
> It keeps all the WER/CER callback, early stopping, and generation logic intact  
> while Ray handles GPU placement, checkpoint routing, and fault tolerance.


In [ ]:
def train_loop_per_worker(config: dict):
    """
    Ray Train worker — one process per GPU.

    Raw training loop (no Seq2SeqTrainer / Accelerate) to avoid the
    PEFT + DDP "got multiple values for keyword argument \'input_ids\'"
    collision.  All model calls use exclusively keyword arguments.
    """
    import os, gc, math
    import torch
    import torch.distributed as dist
    import numpy as np
    import evaluate as hf_evaluate
    from pathlib import Path
    from dataclasses import dataclass
    from typing import Any, Dict, List

    # ── FIX 2: set HF cache dirs BEFORE any HuggingFace import touches disk ──
    # Ray workers inherit a minimal env; without these they default to ~/.cache
    # which is on the small /home partition (typically 99% full on HPC nodes).
    # HF_HOME overrides all HuggingFace cache locations in one variable.
    hf_home = config.get("hf_home", os.path.expanduser("~/.cache/huggingface"))
    os.environ["HF_HOME"]              = hf_home
    os.environ["TRANSFORMERS_CACHE"]   = os.path.join(hf_home, "hub")
    os.environ["HF_DATASETS_CACHE"]    = os.path.join(hf_home, "datasets")
    os.environ["HUGGINGFACE_HUB_CACHE"]= os.path.join(hf_home, "hub")
    # Prevent HF from downloading tokenizer/processor JSON inside worker —
    # we already have them on disk from the driver; set offline mode.
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"]  = "1"
    # Audio backend
    os.environ["FORCE_SOUNDFILE"]      = "1"
    os.environ["PYTORCH_ALLOC_CONF"]   = "expandable_segments:True"

    from datasets import load_from_disk
    from torch.utils.data import DataLoader, DistributedSampler
    from torch.optim import AdamW
    from transformers import (
        WhisperProcessor,
        WhisperForConditionalGeneration,
        get_cosine_schedule_with_warmup,
    )
    from peft import LoraConfig, TaskType, get_peft_model
    import ray.train
    from ray.train.torch import get_device

    # ── Unpack config ───────────────────────────────────────────────────────
    model_name       = config["model_name"]
    dataset_dir      = config["dataset_dir"]
    output_dir       = config["output_dir"]
    lora_r           = config["lora_r"]
    lora_alpha       = config["lora_alpha"]
    lora_dropout     = config["lora_dropout"]
    lora_targets     = config["lora_target_modules"]
    per_gpu_batch    = config["per_gpu_batch"]
    grad_accum       = config["grad_accum"]
    learning_rate    = config["learning_rate"]
    warmup_steps     = config["warmup_steps"]
    epochs           = config["epochs"]
    eval_steps       = config["eval_steps"]
    push_to_hub      = config["push_to_hub"]
    hub_model_id     = config["hub_model_id"]
    amharic_char_map = config["amharic_char_map"]
    punct            = config["punct"]

    device     = get_device()
    world_size = ray.train.get_context().get_world_size()
    rank       = ray.train.get_context().get_world_rank()
    local_rank = ray.train.get_context().get_local_rank()

    DTYPE = torch.bfloat16   # single source of truth for compute dtype

    def normalize_amharic(text):
        for s, t in amharic_char_map.items():
            text = text.replace(s, t)
        text = text.translate(str.maketrans("", "", punct))
        return " ".join(text.split()).strip()

    # ── Dataset ─────────────────────────────────────────────────────────────
    dataset  = load_from_disk(dataset_dir)
    train_ds = dataset["train"]
    val_ds   = dataset["validation"]

    # ── Processor (loaded from local cache — offline mode is set above) ─────
    processor = WhisperProcessor.from_pretrained(
        model_name, language="amharic", task="transcribe",
        local_files_only=True,
    )
    pad_token_id           = processor.tokenizer.pad_token_id
    decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(
        "<|startoftranscript|>"
    )

    # ── Data collator ────────────────────────────────────────────────────────
    @dataclass
    class DataCollator:
        processor: Any
        decoder_start_token_id: int

        def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
            input_feats = [{"input_features": f["input_features"]} for f in features]
            label_feats = [{"input_ids":       f["labels"]}        for f in features]
            batch  = self.processor.feature_extractor.pad(input_feats, return_tensors="pt")
            labels = self.processor.tokenizer.pad(label_feats,  return_tensors="pt")
            lids   = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
            # Strip leading decoder_start token if present (Trainer convention)
            if (lids[:, 0] == self.decoder_start_token_id).all().cpu().item():
                lids = lids[:, 1:]
            batch["labels"] = lids
            return batch

    collator = DataCollator(
        processor=processor,
        decoder_start_token_id=decoder_start_token_id,
    )

    # ── Samplers & loaders ───────────────────────────────────────────────────
    train_sampler = DistributedSampler(
        train_ds, num_replicas=world_size, rank=rank,
        shuffle=True, drop_last=True,
    )
    val_sampler = DistributedSampler(
        val_ds, num_replicas=world_size, rank=rank,
        shuffle=False, drop_last=False,
    )
    train_loader = DataLoader(
        train_ds, batch_size=per_gpu_batch, sampler=train_sampler,
        collate_fn=collator, num_workers=4, pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=per_gpu_batch, sampler=val_sampler,
        collate_fn=collator, num_workers=2, pin_memory=True,
    )

    # ── Build PEFT / LoRA model ──────────────────────────────────────────────
    base = WhisperForConditionalGeneration.from_pretrained(
        model_name,
        torch_dtype=DTYPE,
        local_files_only=True,
    )
    base.config.use_cache              = False
    base.generation_config.language    = "amharic"
    base.generation_config.task        = "transcribe"
    base.generation_config.forced_decoder_ids = None

    lora_cfg = LoraConfig(
        #task_type=TaskType.SEQ_2_SEQ_LM,
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=lora_targets,
        lora_dropout=lora_dropout,
        bias="none",
        modules_to_save=["embed_tokens", "proj_out"],
        inference_mode=False,
    )
    model = get_peft_model(base, lora_cfg)
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable()
    model = model.to(device)

    # ── DDP — manual wrap, no Accelerate ────────────────────────────────────
    if world_size > 1:
        model = torch.nn.parallel.DistributedDataParallel(
            model,
            device_ids=[local_rank],
            output_device=local_rank,
            find_unused_parameters=True,   # required for PEFT modules_to_save
        )

    def unwrap(m):
        """Reach through optional DDP wrapper to the underlying PEFT model."""
        return m.module if isinstance(
            m, torch.nn.parallel.DistributedDataParallel
        ) else m

    # ── Optimizer & LR scheduler ─────────────────────────────────────────────
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate,
        weight_decay=0.01,
    )
    total_steps  = math.ceil(len(train_loader) / grad_accum) * epochs
    warmup       = min(warmup_steps, total_steps // 10)
    lr_scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup,
        num_training_steps=total_steps,
    )

    # ── Metrics ──────────────────────────────────────────────────────────────
    wer_metric = hf_evaluate.load("wer")
    cer_metric = hf_evaluate.load("cer")

    # ── Evaluation function ───────────────────────────────────────────────────
    def run_eval(loader) -> dict:
        """Run full validation pass; return eval_loss, wer, cer."""
        unwrap(model).eval()
        all_pred, all_ref  = [], []
        total_loss, n_seen = 0.0, 0
        total_batches = len(loader)
        if rank == 0:
            print(f"\n🔍 Starting evaluation pass ({total_batches} batches total)...")

        with torch.no_grad():
            for i, batch in enumerate(loader):
                
                if rank == 0 and i % 5 == 0:
                    print(f"      [Eval] Processing batch {i}/{total_batches}...")
                # ── FIX 4: cast input_features to model dtype (bfloat16) ──────
                # DataLoader outputs float32; the model weights are bfloat16.
                # Feeding float32 into a bfloat16 conv layer raises:
                #   RuntimeError: Input type (float) and bias type (BFloat16)
                #   should be the same
                input_features = batch["input_features"].to(device, dtype=DTYPE)
                labels         = batch["labels"].to(device)

                # Forward — keyword-only to avoid PEFT/DDP arg-routing collision
                out = unwrap(model)(
                    input_features=input_features,
                    labels=labels,
                )
                total_loss += out.loss.item()
                n_seen     += 1

                # Generate transcriptions (keyword-only)
                gen_ids = unwrap(model).generate(
                    input_features=input_features,
                    language="amharic",
                    task="transcribe",
                    max_new_tokens=225,
                )

                pred_str = [
                    normalize_amharic(s)
                    for s in processor.tokenizer.batch_decode(
                        gen_ids, skip_special_tokens=True
                    )
                ]
                lab_copy = labels.clone()
                lab_copy[lab_copy == -100] = pad_token_id
                ref_str = [
                    normalize_amharic(s)
                    for s in processor.tokenizer.batch_decode(
                        lab_copy, skip_special_tokens=True
                    )
                ]
                all_pred.extend(pred_str)
                all_ref.extend(ref_str)

        avg_loss = total_loss / max(n_seen, 1)
        if rank == 0:
            print("🧮 Computing final WER/CER metrics...")
            
        wer = wer_metric.compute(predictions=all_pred, references=all_ref) * 100
        cer = cer_metric.compute(predictions=all_pred, references=all_ref) * 100

        unwrap(model).train()
        return {
            "eval_loss": round(avg_loss, 4),
            "wer":       round(wer, 2),
            "cer":       round(cer, 2),
        }

    # ── Training loop ─────────────────────────────────────────────────────────
    global_step         = 0
    best_wer            = float("inf")
    patience_count      = 0
    early_stop_patience = 5
    stop_training       = False

    os.makedirs(output_dir, exist_ok=True)
    unwrap(model).generation_config.max_length = None
    for epoch in range(epochs):
        if stop_training:
            break

        train_sampler.set_epoch(epoch)
        model.train()
        optimizer.zero_grad()
        running_loss = 0.0

        for step, batch in enumerate(train_loader):
            if stop_training:
                break

            # ── FIX 4: cast to bfloat16 in the training loop too ─────────────
            input_features = batch["input_features"].to(device, dtype=DTYPE)
            labels         = batch["labels"].to(device)

            # Keyword-only forward — prevents input_ids double-kwarg crash
            out  = model(
                input_features=input_features,
                labels=labels,
            )
            loss = out.loss / grad_accum
            loss.backward()
            running_loss += out.loss.item()

            if (step + 1) % grad_accum != 0:
                continue   # accumulate more gradients

            # ── Optimizer step ───────────────────────────────────────────────
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            # ── Logging ─────────────────────────────────────────────────────
            if rank == 0 and global_step % 25 == 0:
                avg_loss_log = running_loss / (25 * grad_accum)
                print(
                    f"  step {global_step:5d} | epoch {epoch+1}/{epochs}"
                    f" | loss {avg_loss_log:.4f}"
                    f" | lr {lr_scheduler.get_last_lr()[0]:.2e}"
                )
                running_loss = 0.0

            # ── Eval + checkpoint + early stopping ───────────────────────────
            if global_step % eval_steps == 0:
                metrics = run_eval(val_loader)
                metrics["step"]  = global_step
                metrics["epoch"] = epoch + 1

                # ── FIX 3: ray.train.report() called on EVERY eval step ───────
                # Must be called by ALL workers (not just rank 0) — Ray uses
                # this to synchronise checkpoints and decide fault tolerance.
                ray.train.report(metrics)

                if rank == 0:
                    print(
                        f"  [eval step {global_step}]"
                        f"  WER={metrics['wer']:.2f}%"
                        f"  CER={metrics['cer']:.2f}%"
                        f"  loss={metrics['eval_loss']:.4f}"
                    )
                    if metrics["wer"] < best_wer:
                        best_wer       = metrics["wer"]
                        patience_count = 0
                        ckpt_dir = Path(output_dir) / f"checkpoint-step-{global_step}"
                        unwrap(model).save_pretrained(str(ckpt_dir))
                        processor.save_pretrained(str(ckpt_dir))
                        print(
                            f"  💾 Best checkpoint saved"
                            f" (WER={best_wer:.2f}%) → {ckpt_dir}"
                        )
                    else:
                        patience_count += 1

                # Broadcast patience_count from rank-0 so all workers stop together
                if world_size > 1:
                    pc_t = torch.tensor(patience_count, dtype=torch.long, device=device)
                    dist.broadcast(pc_t, src=0)
                    patience_count = int(pc_t.item())

                if patience_count >= early_stop_patience:
                    if rank == 0:
                        print(
                            f"  ⏹  Early stopping at step {global_step}"
                            f" — no WER improvement for"
                            f" {early_stop_patience} consecutive evals"
                        )
                    stop_training = True

    # ── Final save (rank-0 only) ─────────────────────────────────────────────
    if rank == 0:
        final_dir = Path(output_dir) / "final"
        unwrap(model).save_pretrained(str(final_dir))
        processor.save_pretrained(str(final_dir))
        print(f"\\n✅ Final model saved to: {final_dir}")

    # Final report — all workers must call this
    ray.train.report({"wer": best_wer, "cer": 0.0, "status": "done"})

print("✅ train_loop_per_worker defined")

✅ train_loop_per_worker defined


## 11. Launch Ray Train

`TorchTrainer` launches one worker per GPU, each running `train_loop_per_worker`.  
DDP gradient all-reduce is handled automatically by `ray.train.torch.prepare_model`.


In [14]:
import os
import ray
from ray.train import RunConfig, ScalingConfig, CheckpointConfig
from ray.train.torch import TorchTrainer

# ── HuggingFace cache path ─────────────────────────────────────────────────
# Point to the NVMe scratch space on your HPC node (large, fast, not /home).
# Change this to wherever whisper-large-v3 was already downloaded by the driver.
HF_HOME = os.environ.get(
    "HF_HOME",
    os.path.join(os.path.expanduser("~"), ".cache", "huggingface")
)
# Make sure the driver also uses the same path before any HF downloads
os.environ["HF_HOME"]               = HF_HOME
os.environ["TRANSFORMERS_CACHE"]    = os.path.join(HF_HOME, "hub")
os.environ["HF_DATASETS_CACHE"]     = os.path.join(HF_HOME, "datasets")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(HF_HOME, "hub")
print(f"HF_HOME : {HF_HOME}")

# ── Ray init ──────────────────────────────────────────────────────────────
if not ray.is_initialized():
    ray.init(
        ignore_reinit_error=True,
        num_cpus=NUM_WORKERS * CPUS_PER_WORKER,
        num_gpus=NUM_WORKERS,
    )
print(f"✅ Ray initialized  | resources: {ray.available_resources()}")

# ── Worker config — everything JSON-serialisable ──────────────────────────
worker_config = {
    "model_name"         : MODEL_NAME,
    "dataset_dir"        : DATASET_DIR,
    "output_dir"         : OUTPUT_DIR,
    "hf_home"            : HF_HOME,          # ← FIX 2: passed to every worker
    "lora_r"             : LORA_R,
    "lora_alpha"         : LORA_ALPHA,
    "lora_dropout"       : LORA_DROPOUT,
    "lora_target_modules": LORA_TARGET_MODULES,
    "per_gpu_batch"      : PER_GPU_BATCH,
    "grad_accum"         : GRAD_ACCUM,
    "learning_rate"      : LEARNING_RATE,
    "warmup_steps"       : WARMUP_STEPS,
    "epochs"             : EPOCHS,
    "eval_steps"         : EVAL_STEPS,
    "save_steps"         : SAVE_STEPS,
    "push_to_hub"        : PUSH_TO_HUB,
    "hub_model_id"       : HUB_MODEL_ID,
    "amharic_char_map"   : AMHARIC_CHAR_MAP,
    "punct"              : PUNCT,
}

scaling_config = ScalingConfig(
    num_workers=NUM_WORKERS,
    use_gpu=USE_GPU,
    resources_per_worker={"GPU": 1, "CPU": CPUS_PER_WORKER},
)

run_config = RunConfig(
    name="whisper-large-amharic-lora",
    storage_path=OUTPUT_DIR,
    checkpoint_config=CheckpointConfig(
        num_to_keep=3,
        checkpoint_score_attribute="wer",
        checkpoint_score_order="min",
    ),
)

ray_trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config=worker_config,
    scaling_config=scaling_config,
    run_config=run_config,
)

print(f"🚀 Launching Ray TorchTrainer with {NUM_WORKERS} GPU worker(s)...")
print(f"   Effective global batch: {PER_GPU_BATCH * NUM_WORKERS * GRAD_ACCUM}")
print()

result = ray_trainer.fit()

print("\\n" + "="*60)
print("✅ Ray Train complete!")
print(f"   Best checkpoints : {result.best_checkpoints}")
print(f"   Final metrics    : {result.metrics}")
print("="*60)

HF_HOME : /tmp/s8345520_cache/huggingface


2026-06-04 22:57:46,386	INFO worker.py:2012 -- Started a local Ray instance.
/home/stud_homes/s8345520/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


✅ Ray initialized  | resources: {'accelerator_type:H200': 1.0, 'node:141.2.89.10': 1.0, 'CPU': 8.0, 'node:__internal_head__': 1.0, 'memory': 372888993792.0, 'object_store_memory': 159809568768.0, 'GPU': 2.0}
🚀 Launching Ray TorchTrainer with 2 GPU worker(s)...
   Effective global batch: 128



(TrainController pid=2707299) Requesting resources: {'GPU': 1, 'CPU': 4} * 2
(TrainController pid=2707299) Attempting to start training worker group of size 2 with the following resources: [{'GPU': 1, 'CPU': 4}] * 2
(PlacementGroupCleaner pid=2707813) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(RayTrainWorker pid=2707833) Setting up process group for: env:// [rank=0, world_size=2]
(TrainController pid=2707299) Started training worker group of size 2: 
(TrainController pid=2707299) - (ip=141.2.89.10, pid=2707833) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=2707299) - (ip=141.2.89.10, pid=2707835) world_rank=1, local_rank=1, node_rank=0
(PlacementGroupCleaner pid=2707813) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.
(PlacementGroupCleaner pid=2707813) Failed to query Ray Train Controller actor state. State API may be temporarily un

\n============================================================
✅ Ray Train complete!
   Best checkpoints : []
   Final metrics    : None


(PlacementGroupCleaner pid=2707813) Failed to query Ray Train Controller actor state. State API may be temporarily unavailable. Continuing to monitor.


In [20]:
import os

main_dir = "/tmp/s8345520_cache/whisper-large-amharic-lora"
final_dir = os.path.join(main_dir, "final")

print("📁 Contents of Main Directory:")
if os.path.exists(main_dir):
    print(os.listdir(main_dir))

print("\n📁 Contents of 'final' Directory:")
if os.path.exists(final_dir):
    print(os.listdir(final_dir))

📁 Contents of Main Directory:
['checkpoint-step-2500', 'checkpoint-step-1000', 'whisper-large-amharic-lora', 'checkpoint-step-3000', 'checkpoint-step-2000', 'final', 'checkpoint-step-400', 'checkpoint-step-600', 'checkpoint-step-800', 'checkpoint-step-3500', 'checkpoint-step-1500', 'checkpoint-step-200', 'checkpoint-step-500']

📁 Contents of 'final' Directory:
['README.md', 'tokenizer_config.json', 'adapter_model.safetensors', 'processor_config.json', 'adapter_config.json', 'tokenizer.json']


## 12. Merge LoRA Weights (optional)

After training, LoRA adapters can be merged back into the base weights  
to produce a standalone model with **no PEFT dependency** at inference time.  
Merged model is ~3 GB on disk vs ~250 MB for adapter-only.


In [21]:
import os
from pathlib import Path
from peft import PeftModel
from transformers import WhisperForConditionalGeneration

MERGE_AND_SAVE = True

if MERGE_AND_SAVE:
    # 1. Point directly to the folder that contains both your adapter files
    ACTUAL_CHECKPOINT_DIR = "/tmp/s8345520_cache/whisper-large-amharic-lora/final"
    
    # 2. Define where the full, merged model will live
    MERGED_DIR = os.path.join(SAFE_NVME_ROOT, "whisper-large-amharic-merged")
    os.makedirs(MERGED_DIR, exist_ok=True)

    print(f"🔀 Loading adapter weights from {ACTUAL_CHECKPOINT_DIR}...")
    base = WhisperForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="cpu",
    )
    
    # Load the adapters cleanly from the final folder
    model_peft = PeftModel.from_pretrained(base, ACTUAL_CHECKPOINT_DIR)
    merged     = model_peft.merge_and_unload()

    print(f"💾 Saving complete merged model to {MERGED_DIR}...")
    merged.save_pretrained(MERGED_DIR, safe_serialization=True)
    processor.save_pretrained(MERGED_DIR)

    size_gb = sum(f.stat().st_size for f in Path(MERGED_DIR).rglob("*") if f.is_file()) / 1e9
    print(f"✅ Merged model successfully saved! ({size_gb:.1f} GB)")
    print(f"   Use path: {MERGED_DIR}")
else:
    MERGED_DIR = os.path.join(SAFE_NVME_ROOT, "whisper-large-amharic-lora/final")
    print("ℹ️ Skipping merge — using adapter-only model for inference")

🔀 Loading adapter weights from /tmp/s8345520_cache/whisper-large-amharic-lora/final...


Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

/home/stud_homes/s8345520/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


💾 Saving complete merged model to /tmp/s8345520_cache/whisper-large-amharic-merged...


/home/stud_homes/s8345520/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:683: UserWarning: Input and output embeddings are no longer tied after merging. Setting `tie_word_embeddings=False` in the model config.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Merged model successfully saved! (3.2 GB)
   Use path: /tmp/s8345520_cache/whisper-large-amharic-merged


## 13. Evaluate on Test Set

In [25]:
import torch
import os
from datasets import load_from_disk
from transformers import (
    WhisperForConditionalGeneration, 
    WhisperProcessor, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)

print("📊 Loading merged model into GPU for test evaluation...")
MERGED_DIR = os.path.join(SAFE_NVME_ROOT, "whisper-large-amharic-merged")

# FIX: Force the model back to standard 32-bit float.
# PyTorch AMP (bf16=True in the Trainer) will handle the optimization safely!
eval_model = WhisperForConditionalGeneration.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.float32, 
)
eval_model = eval_model.to("cuda" if torch.cuda.is_available() else "cpu")
eval_model.eval()

# Clear deprecated settings and set language
eval_model.config.max_length = None
eval_model.generation_config.language = "amharic"
eval_model.generation_config.task = "transcribe"
eval_model.generation_config.forced_decoder_ids = None

eval_processor = WhisperProcessor.from_pretrained(
    MERGED_DIR, language="amharic", task="transcribe"
)

# Load dataset from local NVMe cache
dataset = load_from_disk(DATASET_DIR)

data_collator_eval = DataCollatorSpeechSeq2SeqWithPadding(
    processor=eval_processor,
    decoder_start_token_id=eval_model.config.decoder_start_token_id,
)

eval_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_eval_batch_size=64,   
    predict_with_generate=True,
    generation_max_length=225,
    bf16=True,   # This will now work perfectly with the fp32 model
    fp16=False,
    report_to=["none"],
)

eval_trainer = Seq2SeqTrainer(
    model=eval_model,
    args=eval_args,
    processing_class=eval_processor.feature_extractor,
    data_collator=data_collator_eval,
    compute_metrics=compute_metrics,
)

print("🔮 Running inference pipeline across the test split...")
results = eval_trainer.predict(dataset["test"])

label_ids = results.label_ids
label_ids[label_ids == -100] = eval_processor.tokenizer.pad_token_id

print("🔤 Decoding and normalizing text arrays...")
pred_str  = [normalize_amharic(s) for s in
             eval_processor.tokenizer.batch_decode(results.predictions, skip_special_tokens=True)]
label_str = [normalize_amharic(s) for s in
             eval_processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]

test_wer = wer_metric.compute(predictions=pred_str, references=label_str) * 100
test_cer = cer_metric.compute(predictions=pred_str, references=label_str) * 100

print(f"\n{'='*55}")
print(f"  🎯 Final Test WER : {test_wer:.2f}%")
print(f"  🎯 Final Test CER : {test_cer:.2f}%")
print(f"  Google STT baseline      : ~35% WER")
print(f"  Whisper Large no tuning  : ~30% WER")
print(f"{'='*55}")

📊 Loading merged model into GPU for test evaluation...


Loading weights:   0%|          | 0/1260 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/76 [00:00<?, ?it/s]

🔮 Running inference pipeline across the test split...


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
/

🔤 Decoding and normalizing text arrays...

  🎯 Final Test WER : 56.97%
  🎯 Final Test CER : 42.71%
  Google STT baseline      : ~35% WER
  Whisper Large no tuning  : ~30% WER


## 14. Inference

In [26]:
eval_model.eval()

def transcribe(audio_bytes: bytes) -> str:
    """Transcribe raw audio bytes with the fine-tuned LoRA model."""
    audio_array, sr = sf.read(io.BytesIO(audio_bytes))
    if audio_array.ndim > 1:
        audio_array = audio_array.mean(axis=1)
    audio_array = audio_array.astype(np.float32)
    if sr != 16_000:
        import librosa
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16_000)

    inputs = eval_processor.feature_extractor(
        audio_array, sampling_rate=16_000, return_tensors="pt"
    ).input_features.to(eval_model.device)

    with torch.no_grad():
        ids = eval_model.generate(
            inputs,
            language="amharic",
            task="transcribe",
            num_beams=5,        # beam search at inference; greedy was used during training
        )

    return eval_processor.tokenizer.decode(ids[0], skip_special_tokens=True)

# Demo: one sample per dialect
print("🎤 Sample transcriptions per dialect:\n")
seen = set()
for row in combined:
    d = row["dialect"]
    if d not in seen:
        transcription = transcribe(row["audio"]["bytes"])
        print(f"  [{d.upper():10s}]  Ref : {row['text']}")
        print(f"              Hyp : {transcription}")
        print()
        seen.add(d)
    if len(seen) == 5:
        break


🎤 Sample transcriptions per dialect:

  [GOJJAM    ]  Ref : ወጣቶችን አሳታፊ የሆነ ስራ ለመስራት በአካቢያችን/በቀበሌአችን ወጣቶችን አሳታፊ የሆነ ማህበር(ክበብ) በማቋቋም አካባቢያችን ተሰማርተው ይገኛሉ ለዚህ መልካም ስራቸው እንደ አርያ በመቁጠር የአካባቢዉ ሰዎች ለልጆቻቸው የወጣቶችን አርያ እንዲከተሉ የማስተማር እና የአካባቢው ሰው በሚሰባሰብበት ቦታ(አጋጣሚ) ላይ ደግ ስራ ለሰሩት የወጣቶን የሰሩትን በጎ ስራቸውን በመጥቀስ ምስጋና እና እውቅና ይሰጣል።
              Hyp : ወጣቶችን አሣታፊ የሆነ ሥራ ለመሥራት በአካባቢያችን ወይም በቀበሌያችን ወጣቶችን አሣታፊ የሆነ ማህበር ወይም ክብር በማቋቋም አካባቢያችን ተሠማርቶ ይገኛሉ ለዚህ መልካም ሥራቸው እንደ አርያ በመቁጠር የአካባቢው ሠዎች ለልጆቻቸው የውጣቶችን አርያ እንዲከተሉ የማሥተማርና የአካባቢውን ሠው

  [GONDER    ]  Ref : ቴክኖሎጅ እየሆነ እየተስፋፋ መምጣቱን ውደውታል
              Hyp : ቴክኖሎጂ እየሆነ የተሥፋፋ መምጣትን ወደውታል

  [WELLO     ]  Ref : tmket hulum yekrstna emnet teketayoche begara bemehon slemnakebrew
              Hyp : tmket hulum yekrstna emnet teketayoche begara bemehon slemnakebrew

  [SHEWA     ]  Ref : የለም, ቢኖርም አኔ አስካሁን አላወኩባቸውም 
              Hyp : የለም ቢኖርም እኔ እሥካሁን አላወኩባቸውም

  [STANDARD  ]  Ref : የተለያዩ የ ትግራይ አውራጃ ተወላጆች ገንዘባቸው ን አዋጥ ተው የ ልማት ተቋማትን እንዲ መሰርቱ ትልማ አይ ፈቅድ ም
           

## 15. Plot Training Curves

In [27]:
%pip install matplotlib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 854.8 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 1.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 27.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.6/362.6 kB 2.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 21.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 11.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.8/122.8 kB 1.2 MB/s eta 0:00:00ta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [36]:
!kill 3922646
# Load the TensorBoard notebook extension
%reload_ext tensorboard

# Point TensorBoard to your logs directory
# (If your logs are inside the output dir, change this to "nvme_workspace_cache/whisper_large_amharic_lora/runs")
%tensorboard --logdir nvme_workspace_cache/logs

Reusing TensorBoard on port 6006 (pid 3908752), started 0:11:35 ago. (Use '!kill 3908752' to kill it.)

In [29]:
import json, matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path

# Match your specific cache directory from the error log
SAFE_NVME_ROOT = "/tmp/s8345520_cache"
RAY_RESULTS_DIR = os.path.join(SAFE_NVME_ROOT, "ray_results")
OUTPUT_DIR = os.path.join(SAFE_NVME_ROOT, "whisper-large-amharic-lora")

# Search in BOTH Output Dir and Ray Results
search_paths = [Path(OUTPUT_DIR), Path(RAY_RESULTS_DIR)]
candidates = []

for path in search_paths:
    if path.exists():
        candidates.extend(list(path.rglob("trainer_state.json")))

if not candidates:
    print("⚠️ trainer_state.json not found in OUTPUT_DIR or Ray results.")
    print("This means training likely crashed before hitting the first SAVE_STEPS threshold.")
    raise FileNotFoundError("trainer_state.json is completely missing.")

# If multiple are found (from different checkpoints or runs), pick the most recent one
log_path = max(candidates, key=os.path.getmtime)
print(f"✅ Found log at: {log_path}")

with open(log_path) as f:
    trainer_state = json.load(f)

logs        = trainer_state["log_history"]
train_logs  = [l for l in logs if "loss"      in l and "eval_loss" not in l]
eval_logs   = [l for l in logs if "eval_loss" in l]

train_steps = [l["step"]      for l in train_logs]
train_loss  = [l["loss"]      for l in train_logs]
eval_steps  = [l["step"]      for l in eval_logs]
eval_loss   = [l["eval_loss"] for l in eval_logs]
eval_wer    = [l.get("eval_wer", 0)  for l in eval_logs]
eval_cer    = [l.get("eval_cer", 0)  for l in eval_logs]

if not train_steps or not eval_steps:
    print("⚠️ The JSON file was found, but it doesn't contain both train and eval steps yet.")

else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Whisper Large v3 + LoRA  |  Amharic — 4 Leyu Dialects + ALFFA",
                 fontsize=13, fontweight="bold")

    ax1.plot(train_steps, train_loss, color="#3266ad", lw=2, label="train loss")
    ax1.plot(eval_steps,  eval_loss,  color="#e07b39", lw=2, label="val loss",
             marker="o", markersize=4)
    ax1.axvline(x=eval_steps[np.argmin(eval_loss)], color="gray",
                ls="--", alpha=0.6, label="best val")
    ax1.set_title("Loss"); ax1.set_xlabel("Step"); ax1.set_ylabel("Loss")
    ax1.legend(); ax1.grid(alpha=0.3); ax1.set_ylim(bottom=0)

    ax2.plot(eval_steps, eval_wer, color="#9b59b6", lw=2, label="WER %",
             marker="o", markersize=4)
    ax2.plot(eval_steps, eval_cer, color="#27ae60", lw=2, label="CER %",
             marker="s", markersize=4)
    ax2.axhline(y=35, color="#e24b4a", ls="--", lw=1.5, label="Google STT (~35%)")
    ax2.axhline(y=30, color="#888888", ls=":",  lw=1.5, label="Whisper Large (~30%)")

    best = np.argmin(eval_wer)
    ax2.annotate(f"best: {eval_wer[best]:.1f}%",
                 xy=(eval_steps[best], eval_wer[best]),
                 xytext=(eval_steps[best] + (max(eval_steps)*0.1), eval_wer[best] + 5),
                 arrowprops=dict(arrowstyle="->", color="#9b59b6"),
                 fontsize=9, color="#9b59b6")

    ax2.set_title("WER and CER"); ax2.set_xlabel("Step"); ax2.set_ylabel("Error Rate (%)")
    ax2.legend(); ax2.grid(alpha=0.3); ax2.set_ylim(0, 100)

    plt.tight_layout()
    out_png = Path(OUTPUT_DIR) / "training_curves_lora.png"
    plt.savefig(str(out_png), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved plot to {out_png}")

⚠️ trainer_state.json not found in OUTPUT_DIR or Ray results.
This means training likely crashed before hitting the first SAVE_STEPS threshold.


FileNotFoundError: trainer_state.json is completely missing.

## 16. Export Model

In [ ]:
import zipfile, shutil

zip_path = os.path.join(BASE_DIR, "whisper-large-amharic-lora.zip")
src_dir  = MERGED_DIR if MERGE_AND_SAVE else OUTPUT_DIR

if Path(zip_path).exists():
    os.remove(zip_path)

print(f"📦 Zipping {src_dir} → {zip_path}  (excluding checkpoints)...")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in Path(src_dir).rglob("*"):
        if f.is_file() and "checkpoint" not in str(f):
            zf.write(f, f.relative_to(src_dir))

size_gb = Path(zip_path).stat().st_size / 1e9
print(f"✅ Zipped: {size_gb:.2f} GB  →  {zip_path}")

print("\n📋 Contents:")
with zipfile.ZipFile(zip_path, "r") as zf:
    for name in sorted(zf.namelist()):
        mb = zf.getinfo(name).file_size / 1e6
        print(f"   {name:55s} {mb:6.1f} MB")
